In [56]:
import pandas as pd

df = pd.read_csv("../data/raw_weather/varginha/INMET_SE_MG_A515_VARGINHA_01-01-2018_A_31-12-2018.CSV", sep=';')

# limpar nomes das colunas
df.columns = df.columns.str.strip()

df.head()

,DATA (YYYY-MM-DD),HORA (UTC),"PRECIPITA��O TOTAL, HOR�RIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESS�O ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESS�O ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m�),"TEMPERATURA DO AR - BULBO SECO, HORARIA (�C)",TEMPERATURA DO PONTO DE ORVALHO (�C),TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C),TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (�C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (�C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRE��O HORARIA (gr) (� (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2018-01-01,00:00,",4","908,3","908,3","907,9",-9999,"19,7","18,1","20,7","19,5","18,1","17,3",91,81,91,333,"4,1",",1",NaN
1,2018-01-01,01:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN
2,2018-01-01,02:00,0,"908,6","908,7","908,5",-9999,"18,5","17,6","19,6","18,5","18,1","17,5",94,91,94,19,"1,7",",6",NaN
3,2018-01-01,03:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN
4,2018-01-01,04:00,0,"907,4",908,"907,4",-9999,"18,6","17,6","18,6","18,1","17,7","17,2",95,94,94,354,"3,5",",6",NaN


In [57]:
df = df[[
    'DATA (YYYY-MM-DD)',
    'HORA (UTC)',
    'PRECIPITA��O TOTAL, HOR�RIO (mm)',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (�C)',
    'TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C)',
    'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)'
]]

In [58]:
df = df.rename(columns={
    'DATA (YYYY-MM-DD)': 'data',
    'HORA (UTC)': 'hora',
    'PRECIPITA��O TOTAL, HOR�RIO (mm)': 'chuva',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (�C)': 'temp',
    'TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C)': 'temp_max',
    'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)': 'temp_min',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade'
})

df

,data,hora,chuva,temp,temp_max,temp_min,umidade
0,2018-01-01,00:00,",4","19,7","20,7","19,5",91
1,2018-01-01,01:00,-9999,-9999,-9999,-9999,-9999
2,2018-01-01,02:00,0,"18,5","19,6","18,5",94
3,2018-01-01,03:00,-9999,-9999,-9999,-9999,-9999
4,2018-01-01,04:00,0,"18,6","18,6","18,1",94
...,...,...,...,...,...,...,...
8755,2018-12-31,19:00,0,"28,5","28,5","27,9",52
8756,2018-12-31,20:00,0,"28,4","29,7","28,4",47
8757,2018-12-31,21:00,0,"24,9","28,4","24,9",64
8758,2018-12-31,22:00,-9999,-9999,-9999,-9999,-9999


In [59]:
df['datetime'] = pd.to_datetime(df['data'] + ' ' + df['hora'])
df = df.set_index('datetime')

df

,data,hora,chuva,temp,temp_max,temp_min,umidade
datetime,,,,,,,
2018-01-01 00:00:00,2018-01-01,00:00,",4","19,7","20,7","19,5",91
2018-01-01 01:00:00,2018-01-01,01:00,-9999,-9999,-9999,-9999,-9999
2018-01-01 02:00:00,2018-01-01,02:00,0,"18,5","19,6","18,5",94
2018-01-01 03:00:00,2018-01-01,03:00,-9999,-9999,-9999,-9999,-9999
2018-01-01 04:00:00,2018-01-01,04:00,0,"18,6","18,6","18,1",94
...,...,...,...,...,...,...,...
2018-12-31 19:00:00,2018-12-31,19:00,0,"28,5","28,5","27,9",52
2018-12-31 20:00:00,2018-12-31,20:00,0,"28,4","29,7","28,4",47
2018-12-31 21:00:00,2018-12-31,21:00,0,"24,9","28,4","24,9",64


In [60]:
for col in ['chuva', 'temp', 'temp_max', 'temp_min', 'umidade']:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '.')
        .astype(float)
    )

In [61]:
df = df.replace([-9999, -9999.0], pd.NA)
df = df.dropna()

In [62]:
df_daily = df.resample('D').agg({
    'chuva': 'sum',
    'temp': 'mean',
    'temp_max': 'max',
    'temp_min': 'min',
    'umidade': 'mean'
})

df_daily.to_csv('data/processed_weather/vargina_2018.csv')